# FlyLM-3-overfit-test-inference

Load a run from **FlyLM-3-overfit-test-training.ipynb**, present A/B/C images, and inspect
the actual pen drawings and retinal → neural → motor activity in the inherited offline
3D replay. No model training or graph download is needed here.

This is a **same-letter copy test**. Every image exposure resets neural and retinal state,
exactly as in training. `LETTERS='ABC'` produces three independent copies, not a language
continuation. Increase `COPIES_PER_LETTER` to feed each actual drawing back to the eyes.
No recognized character or clean glyph replaces a feedback image.

The training notebook can use **two T4 GPUs**; its unwrapped checkpoint loads on a
**single GPU** here (CPU is supported but slower). Upload the entire run folder, including
`graph/`, `visual_pen_assets.pt`, `experiment.json`, `asset_manifest.json` and `best.pt`.
If best weights came from the frozen-circuit readout stage, that phase is reported explicitly.

All model code, anatomical coordinates and viewer assets are embedded. The inherited
two-link leg and displayed fly body are engineered visualizations, not physical anatomy
or muscle simulation. Fitting three canonical glyphs is not evidence of language learning.

In [ ]:
import importlib.util, subprocess, sys, json
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Select a runtime with PyTorch installed (Kaggle GPU recommended).')

def disk_cuda_info():
    # Import torch only in a child process until the installed build is ready.
    probe = subprocess.run([sys.executable, '-c',
        "import torch,json; print(json.dumps({'version':str(torch.__version__),'cap':list(torch.cuda.get_device_capability()) if torch.cuda.is_available() else None,'arch':torch.cuda.get_arch_list()}))"],
        capture_output=True, text=True, check=True)
    return json.loads(probe.stdout.strip().splitlines()[-1])

disk_info = disk_cuda_info()
print('Installed PyTorch:', disk_info)
if disk_info['cap'] == [6, 0] and 'sm_60' not in disk_info['arch']:
    print('Installing the CUDA 12.6 build with P100 support before importing torch.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch==2.10.0+cu126', 'torchvision==0.25.0+cu126', 'torchaudio==2.10.0+cu126',
        '--index-url', 'https://download.pytorch.org/whl/cu126'])
    importlib.invalidate_caches()
    disk_info = disk_cuda_info()
    print('After installation:', disk_info)
    if disk_info['cap'] != [6, 0] or 'sm_60' not in disk_info['arch']:
        raise RuntimeError('P100 support was not verified after installation. Check the pip output and GPU settings.')

# Like flyLM-2: a fresh kernel can continue immediately after installation.
# Do not remove/reload torch modules: native libraries already loaded cannot be safely swapped.
loaded_torch = sys.modules.get('torch')
if (disk_info['cap'] == [6, 0] and loaded_torch is not None
        and 'sm_60' not in loaded_torch.cuda.get_arch_list()):
    raise RuntimeError(
        'An incompatible torch build was already imported before this setup cell. '
        'Its live CUDA libraries cannot be safely replaced in place. '
        'Run this setup before any torch imports in a fresh kernel; no session restart '
        'is required when this setup runs first.'
    )

required={'numpy':'numpy','PIL':'pillow'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import base64, gzip, hashlib, html, io, math, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from PIL import Image, ImageDraw
from IPython.display import HTML, FileLink, display
print('Runtime:',torch.__version__,'CUDA:',torch.version.cuda)

## 1. Select the ABC run

Set `BUNDLE_DIR` to the **extracted run directory**, not its parent dataset folder or ZIP.
Choose `best.pt` or `latest.pt` explicitly. No fallback to a different checkpoint occurs.
The loader verifies graph, image, trace and experiment fingerprints before loading weights.

In [ ]:
BUNDLE_DIR = Path('/kaggle/input/your-abc-run')  # Set to the extracted ABC run folder.
CHECKPOINT_NAME = 'best.pt'
DEVICE = None  # Auto single GPU; 'cpu' is supported.
OUTPUT_ROOT = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd())/'flylm3_abc_replays'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Exact model definition and checked loading

The forward model and rasterizer match the overfit training notebook. DataParallel is
only a training wrapper; it changes neither parameter names nor the inference architecture.
The saved alphabet must be exactly A/B/C and the task must be independent image copying.

In [ ]:
def forward_kinematics(q, lengths):
    q1,q2 = q.unbind(-1); l1,l2 = lengths
    return torch.stack([l1*q1.cos()+l2*(q1+q2).cos(),
                        l1*q1.sin()+l2*(q1+q2).sin()],dim=-1)

def rasterize_trace(trace, size=None):
    size = CFG['image_size'] if size is None else size
    a = np.asarray(trace); oversample=3
    canvas = Image.new('L',(size*oversample,size*oversample),0)
    draw = ImageDraw.Draw(canvas)
    # Fixed display frame, x in [0.25,1.35], y in [-0.55,0.55].
    xy = np.column_stack([(a[:,0]-.25)/1.1, (.55-a[:,1])/1.1])*(size*oversample-1)
    for j in range(1,len(a)):
        if a[j,2] >= 0.5:
            draw.line([tuple(xy[j-1]),tuple(xy[j])],fill=255,width=oversample)
    return np.asarray(canvas.resize((size,size),Image.Resampling.LANCZOS),dtype=np.float32)/255

class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, voltage_gap, slope):
        ctx.save_for_backward(voltage_gap)
        ctx.slope = slope
        return (voltage_gap >= 0).to(voltage_gap.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (gap,) = ctx.saved_tensors
        # 순전파는 이산 사건이며, 역전파에만 빠른 시그모이드의 대리 미분을 쓴다.
        return grad_output / (1 + ctx.slope * gap.abs()).square(), None


def mixed_neuron_tick(v, signal, slow, refractory, current, alpha, threshold,
                      adaptation, spiking_mask, reset_voltage, refractory_ticks, surrogate_slope):
    candidate = v + alpha[0] * (-v + current - adaptation * slow)
    blocked = spiking_mask & (refractory > 0)
    candidate = torch.where(blocked, torch.full_like(candidate, reset_voltage), candidate)
    spikes = SurrogateSpike.apply(candidate - threshold, surrogate_slope)
    spikes = spikes * (spiking_mask & ~blocked).to(candidate.dtype)
    graded = torch.relu(torch.tanh(candidate - threshold))
    release = torch.where(spiking_mask, spikes, graded)
    # 리셋·불응기 사건의 이산 결정에는 별도 대리 미분을 적용하지 않는다.
    fired = spikes.detach() > 0
    v = torch.where(fired, torch.full_like(candidate, reset_voltage), candidate)
    refractory = torch.where(fired, torch.full_like(refractory, float(refractory_ticks)),
                             torch.clamp(refractory - 1, min=0))
    refractory = torch.where(spiking_mask, refractory, torch.zeros_like(refractory)).detach()
    signal = signal + alpha[1] * (release - signal)
    slow = slow + alpha[2] * (release - slow)
    return v, signal, slow, refractory, spikes


def validate_dynamics_config(cfg):
    if cfg['neuron_mode'] not in ('mixed', 'graded_control'):
        raise ValueError('neuron_mode은 mixed 또는 graded_control이어야 한다.')
    if cfg['unknown_activity'] not in ('graded', 'spiking'):
        raise ValueError('미확인 뉴런의 기본값은 graded 또는 spiking이어야 한다.')
    for key in ('spike_threshold_center', 'spike_threshold_radius', 'spike_reset', 'surrogate_slope'):
        if not math.isfinite(cfg[key]):
            raise ValueError(f'유한한 동역학 설정이 필요하다: {key}')
    if (cfg['spike_threshold_radius'] < 0 or cfg['surrogate_slope'] <= 0
            or cfg['spike_threshold_center'] - cfg['spike_threshold_radius'] <= max(0, cfg['spike_reset'])):
        raise ValueError('스파이크 역치는 항상 0과 리셋 전압보다 높아야 한다.')
    if (type(cfg['refractory_ticks']) is not int or cfg['refractory_ticks'] < 0
            or type(cfg['brain_steps']) is not int or cfg['brain_steps'] < 1):
        raise ValueError('불응기는 0 이상의 정수, 토큰당 tick 수는 양의 정수여야 한다.')


class VisualPenLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, dynamics_ids, uv, lamina_nodes, device):
        super().__init__()
        validate_dynamics_config(cfg)
        self.tbptt = cfg['tbptt']
        self.reset_voltage = cfg['spike_reset']
        self.refractory_ticks = cfg['refractory_ticks']
        self.surrogate_slope = cfg['surrogate_slope']
        self.spike_threshold_center = cfg['spike_threshold_center']
        self.spike_threshold_radius = cfg['spike_threshold_radius']
        codes = np.asarray(dynamics_ids)
        if (codes.shape != (cfg['n_neurons'],) or codes.dtype != np.uint8
                or not np.isin(codes, [0, 1, 2]).all()):
            raise ValueError('뉴런 순서에 맞는 uint8 활동 코드 0/1/2가 필요하다.')
        mask = (codes == 2) & (cfg['neuron_mode'] == 'mixed')
        self.register_buffer('spiking_mask', torch.tensor(mask, dtype=torch.bool, device=device), persistent=False)
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        self.cfg = dict(cfg)
        self.register_buffer('retina_grid', torch.tensor(uv*2-1,device=device).view(1,1,-1,2))
        self.register_buffer('lamina_nodes',torch.tensor(lamina_nodes,device=device,dtype=torch.long))
        self.register_buffer('joint_low',torch.tensor(cfg['joint_low'],device=device))
        self.register_buffer('joint_high',torch.tensor(cfg['joint_high'],device=device))
        self.motor_norm = nn.LayerNorm(len(outputs),eps=1e-6,elementwise_affine=False,device=device)
        self.pen_head = nn.Linear(len(outputs),cfg['pen_steps']*3,bias=True,device=device)
        nn.init.normal_(self.pen_head.weight,std=0.01)
        nn.init.zeros_(self.pen_head.bias)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        offset = self.threshold.tanh()[self.type_ids]
        threshold = torch.where(self.spiking_mask,
            self.spike_threshold_center + self.spike_threshold_radius * offset, 0.5 * offset)
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return (self.theta.new_zeros(4,batch,self.n),
                self.theta.new_zeros(batch,len(self.inputs)))

    def sample_retina(self, images):
        if images.ndim != 4 or images.shape[1] != 1 or not images.is_floating_point():
            raise ValueError('Only floating-point grayscale images [B,1,H,W] enter the model')
        # Linearize the screen before spatial sampling.
        linear = torch.where(images <= .04045, images/12.92, ((images+.055)/1.055).pow(2.4))
        blur = self.cfg['retinal_blur']
        linear = F.avg_pool2d(linear,kernel_size=blur,stride=1,padding=blur//2)
        return F.grid_sample(linear,self.retina_grid.expand(len(images),-1,-1,-1),
            mode='bilinear',align_corners=True).reshape(len(images),-1)

    def external_drive(self, luminance):
        value = self.cfg['retinal_gain']*luminance/(self.cfg['retinal_half_saturation']+luminance)
        return luminance.new_zeros(len(luminance),self.n).index_add(1,self.inputs,value)

    def advance(self, image, neural, retina, weights, alpha, threshold, adaptation):
        v,signal,slow,refractory = neural.unbind(0)
        luminance = self.sample_retina(image)
        eye_alpha = 1-math.exp(-1/self.cfg['retina_tau_ticks'])
        intrinsic = torch.zeros_like(v)
        intrinsic[:,self.lamina_nodes] = self.cfg['lamina_rest']
        for _ in range(self.k):
            retina = retina + eye_alpha*(luminance-retina)
            message = SparseEdges.apply(signal,weights,self.rowptr,self.src,self.dst,self.block)
            v,signal,slow,refractory,_ = mixed_neuron_tick(v,signal,slow,refractory,
                message+self.external_drive(retina)+intrinsic,alpha,threshold,adaptation,
                self.spiking_mask,self.reset_voltage,self.refractory_ticks,self.surrogate_slope)
        return torch.stack((v,signal,slow,refractory)),retina

    def decode_pen(self, neural):
        return self.decode_motor(neural[1].index_select(1, self.outputs))

    def decode_motor(self, motor_signal):
        motor = self.motor_norm(motor_signal)
        raw = self.pen_head(motor).reshape(len(motor),self.cfg['pen_steps'],3)
        angles = self.joint_low+(self.joint_high-self.joint_low)*raw[...,:2].sigmoid()
        xy = forward_kinematics(angles,self.cfg['link_lengths'])
        return xy,raw[...,2],angles

    def forward(self, images, state=None):
        if images.ndim != 5 or images.shape[2] != 1 or images.shape[1] == 0 or not images.is_floating_point():
            raise ValueError('Expected float image sequence [B,T,1,H,W], never character IDs')
        neural,retina = self.initial_state(len(images)) if state is None else state
        assert neural.shape == (4,len(images),self.n)
        assert retina.shape == (len(images),len(self.inputs))
        weights = self.effective_weights(); alpha,threshold,adaptation = self.dynamics()
        positions,contacts,angles = [],[],[]
        for t in range(images.shape[1]):
            args = (images[:,t],neural,retina,weights,alpha,threshold,adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                neural,retina = checkpoint(self.advance,*args,use_reentrant=False,preserve_rng_state=False)
            else:
                neural,retina = self.advance(*args)
            xy,ink,q = self.decode_pen(neural)
            positions.append(xy); contacts.append(ink); angles.append(q)
        return (torch.stack(positions,1),torch.stack(contacts,1),torch.stack(angles,1)),(neural,retina)

In [ ]:
def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''): digest.update(chunk)
    return digest.hexdigest()

def load_run(directory,checkpoint_name,requested_device=None):
    directory=Path(directory)
    required=['experiment.json','asset_manifest.json','visual_pen_assets.pt',checkpoint_name]
    missing=[name for name in required if not (directory/name).is_file()]
    if missing: raise FileNotFoundError(f'Set BUNDLE_DIR to the saved run directory: {directory}; missing {missing}')
    spec=json.loads((directory/'experiment.json').read_text(encoding='utf-8'))
    manifest=json.loads((directory/'asset_manifest.json').read_text(encoding='utf-8'))
    if spec.get('format')!='flylm3_abc_copy_v1': raise ValueError('A FlyLM-3 ABC overfit run is required.')
    cfg=spec['cfg']
    if (spec['data'].get('alphabet') != list('ABC')
            or spec['data'].get('task') != 'independent_image_to_same_character_trace'
            or not spec['data'].get('reset_state_every_example')):
        raise ValueError('Expected an ABC copy experiment with independent reset examples.')
    if any(spec[k]!=manifest[k] for k in ['cfg','data','dynamics']):
        raise ValueError('Experiment settings disagree with the asset manifest.')
    array_names=['body_ids','crow','src','counts','edge_sign','input_nodes','output_nodes',
        'type_ids','dynamics_ids','uv','lamina_nodes']
    arrays={}
    for name in array_names:
        p=directory/'graph'/(name+'.npy')
        if not p.is_file(): raise FileNotFoundError(p)
        if sha256_file(p)!=manifest['arrays'].get(name): raise ValueError(f'Graph checksum mismatch: {name}')
        arrays[name]=np.load(p,mmap_mode='r',allow_pickle=False)
    n=int(cfg['n_neurons']); a=arrays
    if a['body_ids'].shape!=(n,) or len(np.unique(a['body_ids']))!=n: raise ValueError('Invalid neuron IDs')
    if a['crow'].shape!=(n+1,) or a['crow'][0]!=0 or (np.diff(a['crow'])<0).any(): raise ValueError('Invalid CSR pointers')
    edges=int(a['crow'][-1])
    if any(a[k].shape!=(edges,) for k in ['src','counts','edge_sign']): raise ValueError('Invalid CSR edges')
    if (a['src']<0).any() or (a['src']>=n).any() or (a['counts']<=0).any(): raise ValueError('Invalid edge indices/counts')
    for key in ['input_nodes','output_nodes','lamina_nodes']:
        x=a[key]
        if x.ndim!=1 or (x<0).any() or (x>=n).any() or len(np.unique(x))!=len(x): raise ValueError(f'Invalid ports: {key}')
    if not len(a['input_nodes']) or not len(a['output_nodes']): raise ValueError('Empty sensory/motor selection')
    if np.intersect1d(a['input_nodes'],a['output_nodes']).size: raise ValueError('Overlapping input/output ports')
    if a['uv'].shape!=(len(a['input_nodes']),2) or not np.isfinite(a['uv']).all() or ((a['uv']<0)|(a['uv']>1)).any():
        raise ValueError('Invalid retinal projection')
    if a['type_ids'].shape!=(n,) or a['dynamics_ids'].shape!=(n,): raise ValueError('Invalid dynamics assignment')
    assets=torch.load(directory/'visual_pen_assets.pt',map_location='cpu',weights_only=True)
    images,trace,characters=assets['images'],assets['trace'],assets['chars']
    if (images.dtype!=torch.float32 or images.shape!=(len(characters),1,cfg['image_size'],cfg['image_size'])
            or not torch.isfinite(images).all() or images.min()<0 or images.max()>1): raise ValueError('Invalid saved images')
    if trace.shape!=(len(characters),cfg['pen_steps'],3) or not torch.isfinite(trace).all(): raise ValueError('Invalid saved pen templates')
    if characters!=spec['data']['alphabet'] or len(set(characters))!=len(characters): raise ValueError('Alphabet mismatch')
    if hashlib.sha256(images.numpy().tobytes()).hexdigest()!=manifest['image_sha']: raise ValueError('Saved image hash mismatch')
    if hashlib.sha256(trace.numpy().tobytes()).hexdigest()!=manifest['trace_sha']: raise ValueError('Saved trace hash mismatch')
    identity=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
    saved=torch.load(directory/checkpoint_name,map_location='cpu',weights_only=False)
    if saved.get('format')!='flylm3_abc_copy_v1' or saved.get('identity')!=identity:
        raise ValueError('Checkpoint does not match these graph/settings/visual assets.')
    chosen=torch.device(requested_device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    if chosen.type=='cuda' and not torch.cuda.is_available(): raise RuntimeError('Enable the Kaggle GPU accelerator.')
    loaded=VisualPenLM(cfg,a['crow'],a['src'],a['counts'],a['edge_sign'],a['input_nodes'],a['output_nodes'],
        a['type_ids'],a['dynamics_ids'],a['uv'],a['lamina_nodes'],chosen)
    loaded.load_state_dict(saved['model'],strict=True); loaded.eval(); loaded.requires_grad_(False)
    update=int(saved.get('update',0))
    loaded.checkpoint_phase=saved.get('phase','unknown')
    del saved
    # Check the selected runtime using the actual graph kernel before recording.
    with torch.no_grad():
        probe=SparseEdges.apply(loaded.theta.new_zeros(1,n),loaded.effective_weights(),
            loaded.rowptr,loaded.src,loaded.dst,loaded.block)
        if not torch.isfinite(probe).all(): raise RuntimeError('Graph sparse operation failed.')
    return loaded,cfg,arrays,assets,spec,identity,update

model,CFG,graph,assets,experiment,run_identity,trained_updates=load_run(BUNDLE_DIR,CHECKPOINT_NAME,DEVICE)
device=model.theta.device
glyph_images=assets['images'].to(device)
char_to_index={ch:i for i,ch in enumerate(assets['chars'])}
print(f'Loaded {model.n:,} neurons, {len(model.src):,} edges, {len(model.inputs):,} retinal inputs, '
      f'{len(model.outputs):,} right-foreleg motors; checkpoint update {trained_updates}; {device}.')
print('Checkpoint phase:', model.checkpoint_phase)

## 3. Recheck canonical letters and visual dependence

The targets below are supervision for measurement only. The model receives saved pixels,
and its pen output is matched to the saved templates after inference. Both blank-input
and motor-clamp controls should lose discrimination on the balanced three-letter test.
The global mean-trace MSE sets the same position-error scale as training.

In [ ]:
def template_logits(xy, ink_logits):
    # B*T by C only; no B*T*C*S intermediate allocation.
    a=xy.flatten(-2); b=target_trace[...,:2].flatten(-2)
    distance=(a.square().mean(-1,keepdim=True)+b.square().mean(-1)
              -2*(a@b.T)/a.shape[-1]).clamp_min(0)
    ink=ink_logits.sigmoid(); template_ink=target_trace[...,2]
    distance=distance+CFG['ink_loss_weight']*(ink.square().mean(-1,keepdim=True)+
        template_ink.square().mean(-1)-2*(ink@template_ink.T)/ink.shape[-1]).clamp_min(0)
    return -distance/CFG['template_temperature']



target_trace = assets['trace'].to(device)
chars = assets['chars']
mean_xy = target_trace[..., :2].mean(0)
position_scale = float((target_trace[..., :2]-mean_xy).square().mean())

@torch.no_grad()
def copy_predictions(frames):
    prediction, _ = model(frames)  # Every call starts at zero state.
    return prediction

@torch.no_grad()
def score_copy(prediction):
    xy, ink, _ = prediction
    labels = torch.arange(3, device=device)[:, None]
    guesses = template_logits(xy, ink).argmax(-1)
    mse = (xy-target_trace[:, None, :, :2]).square().mean((-1, -2))
    down = target_trace[:, None, :, 2] >= .5
    ink_recall = int(torch.count_nonzero((ink>=0)[down])) / int(down.sum())
    penup_recall = int(torch.count_nonzero((ink<0)[~down])) / int((~down).sum())
    return dict(predicted=''.join(chars[i] for i in guesses.flatten().tolist()),
        char_accuracy=int(torch.count_nonzero(guesses==labels))/3,
        relative_tip_mse=float(mse.mean()/position_scale),
        ink_recall=ink_recall, penup_recall=penup_recall,
        per_letter_mse={ch: float(mse[i, 0]) for i, ch in enumerate(chars)})

canonical = copy_predictions(glyph_images[:, None])
blank = copy_predictions(torch.zeros_like(glyph_images[:, None]))
hook = model.motor_norm.register_forward_pre_hook(lambda module, args: (torch.zeros_like(args[0]),))
try:
    clamped = copy_predictions(glyph_images[:, None])
finally:
    hook.remove()
COPY_REPORT = dict(checkpoint_phase=model.checkpoint_phase,
    canonical=score_copy(canonical), blank=score_copy(blank), motor_clamped=score_copy(clamped))
print(json.dumps(COPY_REPORT, indent=2))
(OUTPUT_ROOT/'abc_copy_metrics.json').write_text(json.dumps(COPY_REPORT, indent=2), encoding='utf-8')

def png_data(pixels):
    output = io.BytesIO()
    Image.fromarray(np.rint(np.asarray(pixels)*255).clip(0, 255).astype(np.uint8)).save(output, format='PNG')
    return 'data:image/png;base64,'+base64.b64encode(output.getvalue()).decode('ascii')

traces = torch.cat([canonical[0][:, 0], canonical[1][:, 0].sigmoid()[..., None]], -1).cpu().numpy()
panels = []
for i, ch in enumerate(chars):
    panels.append('<td style="text-align:center;padding:12px">'+ch+
        '<br><img alt="Input '+ch+'" width="180" style="max-width:100%" src="'+png_data(assets['images'][i, 0].numpy())+'">'+
        '<br>Actual pen output<br><img alt="Pen output for '+ch+'" width="180" style="max-width:100%" src="'+
        png_data(rasterize_trace(traces[i]))+'"></td>')
COMPARISON_HTML = '<table style="width:100%;table-layout:fixed"><tr>'+''.join(panels)+'</tr></table>'
display(HTML(COMPARISON_HTML))
(OUTPUT_ROOT/'abc_copy_comparison.html').write_text(COMPARISON_HTML, encoding='utf-8')
np.savez(OUTPUT_ROOT/'abc_copy_predictions.npz', traces=traces, joints=canonical[2][:, 0].cpu().numpy())

## 4. Anatomical coordinates and offline renderer

These embedded assets are reused from the existing FlyLM-3 inference notebook. Neuron
identities must match the saved full graph; positions are not fabricated for missing IDs.

In [ ]:
# 오프라인 시각화를 위한 공식 MaleCNS v1.0 좌표를 포함한다.
# 출처 코드 1: 세포체, 2: 실제 SWC 골격 정점, 3: tosoma 주석.
import base64
import hashlib
from pathlib import Path

COORDINATE_SHA256 = '8d3602f8ffdfc30239d98a4c484e59a313bff9ea8558ce718af18037b90c58ca'
COORDINATE_DATA = '...'

def bundled_coordinates(cache_dir):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f'neuron_coordinates_{COORDINATE_SHA256[:12]}.npz'
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != COORDINATE_SHA256:
        raw = base64.b64decode(COORDINATE_DATA)
        if hashlib.sha256(raw).hexdigest() != COORDINATE_SHA256:
            raise ValueError('Bundled coordinate data failed SHA-256 verification')
        temporary = path.with_suffix('.partial')
        temporary.write_bytes(raw)
        temporary.replace(path)
    return path

In [ ]:
THREE_SHA256 = '170c6789f43217c96b3170f4b42fafe135de7f7cd48497a4218f9757ee1d49fa'
THREE_GZIP_B64 = '...'
THREE_LICENSE = 'The MIT License\n\nCopyright © 2010-2023 three.js authors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n'
THREE_JS=gzip.decompress(base64.b64decode(THREE_GZIP_B64)).decode('utf-8')
if hashlib.sha256(THREE_JS.encode()).hexdigest()!=THREE_SHA256: raise ValueError('Embedded renderer hash mismatch')
print('Offline renderer ready: three.js 0.160.1 / MIT')

In [ ]:
def prepare_layout(model,graph,coordinate_file):
    with np.load(coordinate_file,allow_pickle=False) as stored:
        meta=json.loads(str(stored['metadata']))
        if meta['dataset']!='male-cns:v1.0' or meta['units']!='8nm_voxels': raise ValueError('Unexpected coordinate dataset')
        ids=stored['body_ids']; order=np.argsort(ids); at=np.searchsorted(ids[order],graph['body_ids'])
        if (at>=len(ids)).any() or not np.array_equal(ids[order][at],graph['body_ids']):
            raise ValueError('The embedded coordinate archive does not cover this checkpoint; no fake positions are substituted.')
        xyz=stored['xyz'][order[at]].astype(np.float32)
        source=stored['source'][order[at]].astype(np.uint8)
    if xyz.shape!=(model.n,3) or not np.isfinite(xyz).all() or (source==0).any(): raise ValueError('Missing anatomical coordinates')
    center=(xyz.min(0)+xyz.max(0))/2; span=float(np.ptp(xyz,axis=0).max())
    # Preserve relative scale across all axes: render x, -z, y for the two v2 projections.
    normalized=(xyz-center)*(2/span)
    positions=np.column_stack([normalized[:,0],-normalized[:,2],normalized[:,1]]).astype(np.float32)
    roles=np.zeros(model.n,np.uint8);roles[graph['input_nodes']]=1;roles[graph['output_nodes']]=2
    return dict(positions=positions,roles=roles,xyz=xyz,source=source,body_ids=np.array(graph['body_ids']))

layout=prepare_layout(model,graph,bundled_coordinates(OUTPUT_ROOT/'coordinate_cache'))
print('Anatomical positions:',len(layout['positions']),'; provenance counts:',
      dict(zip(*np.unique(layout['source'],return_counts=True))))

## 5. Record independent copies and optional self-feedback

Each exposure starts with zero neural and retinal state. For each requested letter, the
first exposure uses its canonical saved image. Additional copies use the preceding actual
pen raster, still with a fresh state. This tests drawing feedback without introducing the
carried-state behavior that the overfit model was never trained to use.

Retinal values, synaptic signals, spike counts, joint angles and pen contact are recorded
from real model computations. Playback speed is independent of dimensionless neural ticks.

In [ ]:
class ObservedVisualPenLM(VisualPenLM):
    observer = None
    def advance(self, image, neural, retina, weights, alpha, threshold, adaptation):
        v,signal,slow,refractory = neural.unbind(0)
        luminance = self.sample_retina(image)
        eye_alpha = 1-math.exp(-1/self.cfg['retina_tau_ticks'])
        intrinsic = torch.zeros_like(v)
        intrinsic[:,self.lamina_nodes] = self.cfg['lamina_rest']
        for observed_tick in range(1,self.k+1):
            retina = retina + eye_alpha*(luminance-retina)
            message = SparseEdges.apply(signal,weights,self.rowptr,self.src,self.dst,self.block)
            v,signal,slow,refractory,spikes = mixed_neuron_tick(v,signal,slow,refractory,
                message+self.external_drive(retina)+intrinsic,alpha,threshold,adaptation,
                self.spiking_mask,self.reset_voltage,self.refractory_ticks,self.surrogate_slope)
            if self.observer is not None:
                self.observer(observed_tick,signal,retina,spikes)
        return torch.stack((v,signal,slow,refractory)),retina


def capture_schedule(observations,ticks,budget):
    if budget<2*observations:
        raise ValueError(f'MAX_NEURAL_FRAMES must be at least {2*observations} for this prompt/output length.')
    selected=[{0,ticks} for _ in range(observations)]
    interiors=observations*max(ticks-1,0)
    count=min(budget-2*observations,interiors)
    if count:
        for index in np.linspace(0,interiors-1,count,dtype=int):
            observation,offset=divmod(int(index),ticks-1)
            selected[observation].add(offset+1)
    return [sorted(x) for x in selected]

@torch.no_grad()
def record_abc_copy(model,prompt,copies_per_letter,max_frames=240,look_seconds=.6,draw_seconds=2.,max_memory_mib=384):
    if not isinstance(prompt,str) or not prompt: raise ValueError('A nonempty prompt is required.')
    missing=sorted(set(prompt)-set(char_to_index))
    if missing: raise ValueError(f'Prompt has characters outside the saved visual alphabet: {missing!r}')
    if type(copies_per_letter)!=int or copies_per_letter<1: raise ValueError('copies_per_letter must be a positive integer.')
    if look_seconds<=0 or draw_seconds<=0: raise ValueError('Replay durations must be positive.')
    observations=len(prompt)*copies_per_letter
    schedule=capture_schedule(observations,model.k,max_frames)
    frame_count=sum(map(len,schedule)); estimated=frame_count*(model.n*4+len(model.inputs)*2)
    if estimated>max_memory_mib*1024**2:
        raise ValueError(f'Replay arrays would use {estimated/1024**2:.1f} MiB; lower MAX_NEURAL_FRAMES or output length.')
    signals=[];spike_frames=[];retinas=[];frames=[];images=[];feedback=[];traces=[];joints=[]
    obs_meta=[];outputs=[];events=[];clock=0.;global_ticks=0
    state=model.initial_state(1);image=None
    previous_training=model.training;model.eval()
    # Swap only the method implementation on this instance, restoring it in finally.
    original_advance=model.advance
    had_override='advance' in model.__dict__;old_override=model.__dict__.get('advance')
    old_observer=getattr(model,'observer',None);had_observer=hasattr(model,'observer')
    import types
    model.advance=types.MethodType(ObservedVisualPenLM.advance,model)
    weights=model.effective_weights();alpha,threshold,adaptation=model.dynamics()
    try:
        for oi in range(observations):
            letter_index, copy_index = divmod(oi, copies_per_letter)
            state = model.initial_state(1)
            if copy_index == 0: image=glyph_images[char_to_index[prompt[letter_index]]][None]
            else: image=torch.from_numpy(feedback[-1].astype(np.float32)/255).to(device)[None,None]
            # The inference image is exactly the saved pixel bank / PIL feedback, before any retinal filtering.
            pixels=image[0,0].mul(255).round().to(torch.uint8).cpu().numpy()
            images.append(pixels)
            frame_indices=[];frame_ticks=[];accum=model.theta.new_zeros(model.n)
            selected=set(schedule[oi]);base_tick=global_ticks
            def snapshot(tick,signal,retina):
                if not torch.isfinite(signal).all() or not torch.isfinite(retina).all(): raise RuntimeError('Nonfinite neural activity')
                if accum.max()>65535: raise RuntimeError('Spike-count interval exceeds uint16 capacity')
                signals.append(signal[0].clamp(0,1).mul(65535).round().to(torch.uint16).cpu().numpy())
                spike_frames.append(accum.to(torch.uint16).cpu().numpy())
                retinas.append(retina[0].clamp(0,1).mul(65535).round().to(torch.uint16).cpu().numpy())
                frame_indices.append(len(frames));frame_ticks.append(tick)
                frames.append(dict(global_tick=base_tick+tick,observation=oi,tick=tick))
                accum.zero_()
            def observe(tick,signal,retina,spikes):
                accum.add_(spikes[0])
                if tick in selected: snapshot(tick,signal,retina)
            model.observer=observe
            snapshot(0,state[0][1],state[1])
            state=model.advance(image,*state,weights,alpha,threshold,adaptation)
            global_ticks+=model.k
            obs_meta.append(dict(source='prompt' if copy_index==0 else 'pen_feedback', letter=prompt[letter_index], copy_index=copy_index,start=clock,
                frames=frame_indices,ticks=frame_ticks))
            events.append(dict(kind='look',start=clock,duration=look_seconds,observation=oi,output=len(outputs)-1))
            clock+=look_seconds
            if True:  # Every independent exposure produces one same-letter drawing.
                xy,ink,q=model.decode_pen(state[0])
                trace=torch.cat([xy[0],ink[0].sigmoid()[:,None]],-1).cpu().numpy()
                if not np.isfinite(trace).all(): raise RuntimeError('Nonfinite pen trajectory')
                output_index=len(outputs);traces.append(trace);joints.append(q[0].cpu().numpy())
                # Same rasterizer as training. It produces exact 8-bit PIL-derived pixels.
                rendered=rasterize_trace(trace,model.cfg['image_size'])
                feedback.append(np.rint(rendered*255).astype(np.uint8))
                outputs.append(dict(start=clock,observation=oi))
                events.append(dict(kind='draw',start=clock,duration=draw_seconds,observation=oi,output=output_index))
                clock+=draw_seconds
            if (oi+1)%8==0: print(f'Recorded {oi+1}/{observations} exposures, {global_ticks} neural ticks.',flush=True)
    finally:
        if had_override: model.advance=old_override
        else: del model.advance
        if had_observer: model.observer=old_observer
        else: del model.observer
        model.train(previous_training)
    return dict(signal=np.stack(signals),spikes=np.stack(spike_frames),retina=np.stack(retinas),
        images=np.stack(images),feedback=np.stack(feedback),traces=np.stack(traces),joints=np.stack(joints),
        frames=frames,observations=obs_meta,outputs=outputs,events=events,duration=clock,
        neural_ticks=global_ticks,brain_steps=model.k,recorded_frames=len(frames),prompt=prompt,
        identity=run_identity,checkpoint=CHECKPOINT_NAME,trained_updates=trained_updates, copies_per_letter=copies_per_letter,
        reset_state_every_exposure=True, checkpoint_phase=model.checkpoint_phase)

## 6. Replay builder

The inherited offline 3D viewer shows the presented image, mapped retinal response,
neural activity, pen trajectory and feedback raster. Use Play, seek and the view controls.
The saved HTML embeds its recording and assets and can be opened without a server.

In [ ]:
PLAYER_HTML = '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>FlyLM-3 · Eyes, neurons & pen</title><link rel="icon" href="data:,">\n<style>\n:root{color-scheme:light;--ink:#17232f;--muted:#61717e;--line:#dbe3e8;--paper:#f3f6f8;--accent:#147c8c}\n*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--ink);font:14px/1.5 system-ui,-apple-system,sans-serif}\nmain{max-width:1480px;margin:auto;padding:24px}h1{font-size:26px;font-weight:650;letter-spacing:-.6px;margin:0}h2{font-size:15px;font-weight:650;margin:0}.sub{color:var(--muted);font-size:12px;margin:4px 0}.top{display:flex;justify-content:space-between;align-items:center;gap:16px;margin-bottom:18px}.tag{font-size:11px;letter-spacing:.09em;text-transform:uppercase;color:var(--accent);font-weight:700}.card{background:white;border:1px solid var(--line);border-radius:12px;overflow:hidden;margin-top:16px}.bar{padding:13px 16px;display:flex;align-items:center;justify-content:space-between;gap:12px;flex-wrap:wrap}.controls{display:flex;gap:10px;flex-wrap:wrap;align-items:center}.controls label{display:flex;gap:6px;align-items:center;color:var(--muted);font-size:12px}\nbutton,select{font:inherit;background:white;color:var(--ink);border:1px solid #c9d4dc;border-radius:7px;padding:7px 11px;cursor:pointer}button:hover{background:#eaf1f5}button:disabled{opacity:.45;cursor:default}button.primary{background:var(--accent);color:white;border-color:var(--accent);min-width:88px}button:focus-visible,select:focus-visible,input:focus-visible{outline:3px solid #31b5c5;outline-offset:2px}input[type=range]{accent-color:var(--accent)}#seek{width:100%;margin:0}#speed{width:115px}#time{font-variant-numeric:tabular-nums;white-space:nowrap}.transport{padding:14px 16px}.transport .controls{margin-bottom:12px}.push{margin-left:auto}#phase{background:#eaf3f4;padding:5px 9px;border-radius:6px;font-size:12px}#load{padding:12px 16px;background:#e6f3f3;border-radius:8px}#error{display:none;white-space:pre-wrap;color:#943f39;background:#fff0ed;border:1px solid #e4bfb8;padding:14px;border-radius:8px}\n#brainStage{position:relative;height:540px;background:#08121e}canvas.webgl{display:block;width:100%;height:100%;touch-action:none}.view-label{position:absolute;top:12px;color:#91a8bb;pointer-events:none;font-size:11px;letter-spacing:.05em}.left-label{left:18px}.right-label{left:calc(50% + 18px)}#brainDivider{position:absolute;left:50%;top:0;bottom:0;border-left:1px solid #233548;pointer-events:none}.legend{display:flex;gap:14px;flex-wrap:wrap;color:var(--muted);font-size:11px;padding:10px 16px}.dot{width:7px;height:7px;border-radius:50%;display:inline-block;margin-right:5px}.lower{display:grid;grid-template-columns:1.1fr 1fr;gap:16px}.lower>.card{margin-top:16px}.lower>.card:first-child{display:flex;flex-direction:column}#flyStage{position:relative;height:460px;min-height:460px;flex:1;background:#eff3f5}.overlay{position:absolute;bottom:12px;left:12px;background:#fffffff0;border:1px solid var(--line);padding:5px 9px;border-radius:6px;font-size:11px;pointer-events:none}.sensor-grid{display:grid;grid-template-columns:1fr 1fr;gap:14px;padding:4px 16px 16px}.sensor{min-width:0}.sensor canvas{display:block;width:100%;aspect-ratio:1;border-radius:7px;background:#08121e}.sensor h3{font-size:11px;text-transform:uppercase;letter-spacing:.055em;font-weight:600;color:var(--muted);margin:5px 0 8px}.sensor .caption{font-size:11px;color:var(--muted);margin-top:7px;min-height:32px}.pen-row{border-top:1px solid var(--line);padding-top:14px}.pen-row canvas{background:#fbfcfd;border:1px solid var(--line)}#gallery{display:grid;grid-template-columns:repeat(auto-fill,minmax(78px,1fr));gap:10px;padding:4px 16px 16px}.thumb{padding:0;overflow:hidden;text-align:left;border:1px solid var(--line);background:white}.thumb canvas{width:100%;aspect-ratio:1;display:block;background:#fff}.thumb span{font-size:10px;display:block;padding:3px 7px;color:var(--muted)}.thumb.future{opacity:.3}.thumb.active{border-color:var(--accent);box-shadow:0 0 0 1px var(--accent)}.notes{font-size:11px;color:var(--muted);padding:10px 0}.stat{font-variant-numeric:tabular-nums;font-size:11px;color:var(--muted)}#brainCard:fullscreen{border-radius:0;margin:0;display:flex;flex-direction:column}#brainCard:fullscreen #brainStage{flex:1;height:auto}.sensor canvas{image-rendering:auto}\n@media(max-width:800px){main{padding:12px}.top{align-items:flex-start}h1{font-size:22px}.lower{grid-template-columns:1fr}#brainStage{height:440px}#flyStage{height:400px;min-height:400px;flex:none}.bar{padding:10px 12px}.push{margin-left:0}.top .stat{display:none}}@media(prefers-reduced-motion:reduce){*{scroll-behavior:auto}}\n</style></head><body><main>\n<div class="top"><div><div class="tag">FlyLM-3 / inference replay</div><h1>Through the eyes. Onto the page.</h1><p class="sub">Recorded neural activity and right-foreleg pen commands, on one timeline.</p></div><div class="stat" id="summary"></div></div>\n<div id="load" role="status">Loading the recorded experiment…</div><div id="error" role="alert"></div>\n<section class="card transport" aria-label="Replay controls">\n <div class="controls"><button id="play" class="primary" disabled>Play</button><button id="previous" aria-label="Previous exposure">←</button><button id="next" aria-label="Next exposure">→</button>\n <label>Replay speed <input id="speed" type="range" min="0.1" max="8" step="0.1" value="1"><output id="speedValue">1.0×</output></label>\n <label><input id="loop" type="checkbox">Loop</label><span id="phase">Ready</span><span id="time" class="push">0:00 / 0:00</span></div>\n <input id="seek" aria-label="Replay timeline" type="range" min="0" max="1" step="0.001" value="0"><div class="sub" id="frameInfo">Replay time is presentation time, not biological time.</div>\n</section>\n<section class="card" id="brainCard"><div class="bar"><div><h2>Neuron activity</h2><div class="sub">Every retained neuron · anatomical coordinates · no spatial downsampling</div></div><div class="controls">\n <label>Activity <select id="activityMode"><option value="mixed">Spikes + graded signal</option><option value="signal">Synaptic signal (all)</option><option value="spikes">Spike events only</option></select></label>\n <label>View <select id="brainView"><option value="dual">Two anatomical views</option><option value="orbit">Orbit in 3D</option></select></label>\n <label>Quality <select id="quality"><option value="1">1×</option><option value="2" selected>2×</option><option value="3">3×</option></select></label>\n <label>Brightness <input id="gain" aria-label="Neural brightness" type="range" min="0.5" max="6" value="2" step="0.1" style="width:70px"></label>\n <button id="fullBrain">Fullscreen</button><button id="saveBrain">Save 4K PNG</button></div></div>\n <div id="brainStage"><canvas id="brain" class="webgl" aria-label="High-resolution anatomical neuron activity"></canvas><span class="view-label left-label" id="brainLabel">X–Z</span><span class="view-label right-label" id="brainLabelRight">X–Y</span><div id="brainDivider"></div></div>\n <div class="legend"><span><i class="dot" style="background:#5ce6f3"></i>Visual receptors</span><span><i class="dot" style="background:#829bff"></i>Other neurons</span><span><i class="dot" style="background:#ffaf63"></i>Right-foreleg motors</span><span class="push" id="activityInfo"></span></div>\n</section>\n<div class="lower"><section class="card"><div class="bar"><div><h2>3D fly · right foreleg</h2><div class="sub">Drag to orbit · scroll to zoom</div></div><button id="resetFly">Reset view</button></div>\n <div id="flyStage"><canvas id="fly" class="webgl" aria-label="3D fly with animated right front leg"></canvas><div class="overlay" id="penState">Pen raised</div></div><div class="legend">Illustrative body. Orange leg follows saved joint angles and pen contact; the other legs remain still.</div>\n</section><section class="card"><div class="bar"><div><h2>What the fly sees / what its leg draws</h2><div class="sub" id="exposureLabel"></div></div></div>\n <div class="sensor-grid"><div class="sensor"><h3>Presented image</h3><canvas id="eye" aria-label="Actual image presented to the fly"></canvas><div class="caption" id="eyeCaption"></div></div>\n <div class="sensor"><h3>Mapped retinal response</h3><canvas id="retina" aria-label="Filtered receptor luminance at mapped eye coordinates"></canvas><div class="caption">Mapped R1–R6 luminance after the model’s spatial and temporal filtering.</div></div>\n <div class="sensor pen-row"><h3>Pen trace · current sample</h3><canvas id="paper" aria-label="What the fly leg is drawing"></canvas><div class="caption">Actual pen-tip path. Ink appears only where the predicted contact is on.</div></div>\n <div class="sensor pen-row"><h3>Feedback image</h3><canvas id="feedback" aria-label="Rasterized pen drawing fed back to the eyes"></canvas><div class="caption">The completed drawing becomes the next input. No recognized text or clean glyph replaces it.</div></div></div>\n</section></div>\n<section class="card"><div class="bar"><h2>What the leg drew</h2><div class="sub">Click a sample to replay it · no text decoding</div></div><div id="gallery"></div></section>\n<div class="notes" id="notes"></div>\n</main>\n<script>__THREE_JS__</script>\n<script id="payload" type="application/json">__PAYLOAD__</script>\n<script>\n\'use strict\';\n(async()=>{\nconst $=id=>document.getElementById(id), data=JSON.parse($(\'payload\').textContent);\nconst fail=e=>{ $(\'error\').style.display=\'block\';$(\'error\').textContent=\'Replay could not open: \'+(e.message||e)+\'\\nOpen the saved HTML in current Chrome or Edge with WebGL enabled.\';$(\'load\').style.display=\'none\';console.error(e);};\ntry{\nif(!window.THREE)throw Error(\'Embedded 3D renderer is unavailable.\');\nif(!window.DecompressionStream)throw Error(\'This browser does not support offline gzip decoding.\');\nasync function unpack(p){const binary=atob(p.data),bytes=new Uint8Array(binary.length);for(let i=0;i<bytes.length;i++)bytes[i]=binary.charCodeAt(i);const buf=await new Response(new Blob([bytes]).stream().pipeThrough(new DecompressionStream(\'gzip\'))).arrayBuffer();const types={u8:Uint8Array,u16:Uint16Array,f32:Float32Array};const result=new types[p.type](buf);if(result.length!==p.shape.reduce((a,b)=>a*b,1))throw Error(\'Invalid replay array size\');return result;}\nconst keys=Object.keys(data.arrays), vals=await Promise.all(keys.map(k=>unpack(data.arrays[k]))), a=Object.fromEntries(keys.map((k,i)=>[k,vals[i]]));\ndelete data.arrays;$(\'payload\').remove();\nconst n=data.neurons,R=data.receptors,H=data.image_size,S=data.pen_steps,O=data.observations.length,C=data.outputs.length,F=data.frames.length;\nconst events=data.events,total=data.duration;\nlet time=0,playing=false,speed=Number($(\'speed\').value),lastRAF=0,dirty=true,lastFrame=-1,lastObservation=-1,lastRetinaFrame=-1,lastMode=\'\',lastDrawKey=\'\',lastGallery=-2;\nlet current={event:0,frame:0,observation:0,output:-1,point:-1,kind:\'look\'};\nconst v=(x,y,z)=>new THREE.Vector3(x,y,z);\nfunction renderer(canvas,bg){const r=new THREE.WebGLRenderer({canvas,antialias:true,alpha:false,preserveDrawingBuffer:true});r.setClearColor(bg);r.outputColorSpace=THREE.SRGBColorSpace;return r;}\nconst brainRenderer=renderer($(\'brain\'),0x08121e),flyRenderer=renderer($(\'fly\'),0xeff3f5);\nconst brainScene=new THREE.Scene(),brainGeometry=new THREE.BufferGeometry();\nbrainGeometry.setAttribute(\'position\',new THREE.BufferAttribute(a.positions,3));\nconst brainExtent=[0,0,0];for(let i=0;i<a.positions.length;i++)brainExtent[i%3]=Math.max(brainExtent[i%3],Math.abs(a.positions[i]));\nconst colors=new Float32Array(n*3),levels=new Float32Array(n),roleColors=[[.51,.61,1],[.36,.90,.95],[1,.68,.37]];\nfor(let i=0;i<n;i++)colors.set(roleColors[a.roles[i]],3*i);\nbrainGeometry.setAttribute(\'color\',new THREE.BufferAttribute(colors,3));brainGeometry.setAttribute(\'level\',new THREE.BufferAttribute(levels,1).setUsage(THREE.DynamicDrawUsage));\nconst brainMaterial=new THREE.ShaderMaterial({transparent:true,depthWrite:false,blending:THREE.AdditiveBlending,\n uniforms:{gain:{value:2},pointScale:{value:2}},\n vertexShader:`attribute vec3 color;attribute float level;varying vec3 tint;varying float light;uniform float gain;uniform float pointScale;void main(){tint=color;light=clamp(level*gain,0.0,1.0);gl_Position=projectionMatrix*modelViewMatrix*vec4(position,1.0);gl_PointSize=pointScale*(1.05+1.7*sqrt(light));}`,\n fragmentShader:`varying vec3 tint;varying float light;void main(){float d=length(gl_PointCoord-vec2(.5));if(d>.5)discard;float edge=1.0-smoothstep(.22,.5,d);gl_FragColor=vec4(tint*(.045+.955*sqrt(light)),edge*(.2+.8*sqrt(light)));}`});\nconst brainPoints=new THREE.Points(brainGeometry,brainMaterial);brainScene.add(brainPoints);\nconst cameraFront=new THREE.OrthographicCamera(-1,1,1,-1,.01,20),cameraSide=cameraFront.clone(),cameraOrbit=new THREE.PerspectiveCamera(42,1,.01,50);\ncameraFront.position.set(0,0,5);cameraFront.lookAt(0,0,0);cameraSide.position.set(0,5,0);cameraSide.up.set(0,0,1);cameraSide.lookAt(0,0,0);\nconst flyScene=new THREE.Scene();flyScene.add(new THREE.HemisphereLight(0xffffff,0x7f8794,2.4));const light=new THREE.DirectionalLight(0xffffff,3);light.position.set(3,-2,6);flyScene.add(light);\nconst flyCamera=new THREE.PerspectiveCamera(40,1,.01,50);flyCamera.up.set(0,0,1);\nfunction orbit(canvas,camera,target,yaw,pitch,radius,enabled=()=>true){let drag=null;const state={yaw,pitch,radius};const initial={...state};function update(){camera.position.copy(target).add(v(state.radius*Math.cos(state.pitch)*Math.cos(state.yaw),state.radius*Math.cos(state.pitch)*Math.sin(state.yaw),state.radius*Math.sin(state.pitch)));camera.lookAt(target);dirty=true;}\n canvas.addEventListener(\'pointerdown\',e=>{if(!enabled())return;drag={x:e.clientX,y:e.clientY};canvas.setPointerCapture(e.pointerId);});canvas.addEventListener(\'pointermove\',e=>{if(!drag)return;state.yaw-=(e.clientX-drag.x)*.007;state.pitch=Math.max(.06,Math.min(1.48,state.pitch+(e.clientY-drag.y)*.007));drag={x:e.clientX,y:e.clientY};update();});const release=()=>drag=null;canvas.addEventListener(\'pointerup\',release);canvas.addEventListener(\'pointercancel\',release);canvas.addEventListener(\'wheel\',e=>{if(!enabled())return;e.preventDefault();state.radius=Math.max(1.3,Math.min(12,state.radius*Math.exp(e.deltaY*.001)));update();},{passive:false});update();return ()=>{Object.assign(state,initial);update();};}\nconst resetFly=orbit($(\'fly\'),flyCamera,v(.35,0,.4),-.9,.7,4.8);cameraOrbit.up.set(0,0,1);orbit($(\'brain\'),cameraOrbit,v(0,0,0),-1.5,.6,3.5,()=>$(\'brainView\').value===\'orbit\');\nconst bodyMat=new THREE.MeshStandardMaterial({color:0x6b4931,roughness:.65,metalness:.18}),darkMat=new THREE.MeshStandardMaterial({color:0x322a26,roughness:.7}),legMat=new THREE.MeshStandardMaterial({color:0xe18a32,roughness:.45}),eyeMat=new THREE.MeshStandardMaterial({color:0x9a342f,roughness:.35}),wingMat=new THREE.MeshStandardMaterial({color:0xd3e8ed,transparent:true,opacity:.52,roughness:.28,side:THREE.DoubleSide});\nconst sphereGeo=new THREE.SphereGeometry(1,32,20),rodGeo=new THREE.CylinderGeometry(1,1,1,12);\nfunction ellipsoid(pos,scale,mat){const o=new THREE.Mesh(sphereGeo,mat);o.position.copy(pos);o.scale.copy(scale);flyScene.add(o);return o;}\nfunction rod(mat,radius=.035){const o=new THREE.Mesh(rodGeo,mat);o.userData.radius=radius;flyScene.add(o);return o;}\nfunction placeRod(o,p,q){o.position.copy(p).add(q).multiplyScalar(.5);const d=q.clone().sub(p);o.quaternion.setFromUnitVectors(v(0,1,0),d.clone().normalize());o.scale.set(o.userData.radius,Math.max(d.length(),.0001),o.userData.radius);}\nellipsoid(v(0,-.56,.65),v(.3,.64,.28),bodyMat);ellipsoid(v(0,.04,.72),v(.31,.36,.3),bodyMat);ellipsoid(v(0,.48,.8),v(.3,.24,.24),darkMat);\nellipsoid(v(-.2,.57,.84),v(.16,.15,.18),eyeMat);ellipsoid(v(.2,.57,.84),v(.16,.15,.18),eyeMat);\nfor(let j=0;j<5;j++){const ring=new THREE.Mesh(new THREE.TorusGeometry(.24-j*.016,.012,8,40),darkMat);ring.rotation.x=Math.PI/2;ring.scale.set(1,1,1);ring.position.set(0,-.27-j*.17,.65);flyScene.add(ring);}\nfor(const side of [-1,1]){const wing=ellipsoid(v(side*.37,-.22,.98),v(.24,.7,.018),wingMat);wing.rotation.z=-side*.35;const antenna=rod(darkMat,.012);placeRod(antenna,v(side*.09,.64,.93),v(side*.17,.9,1.03));}\nfor(const side of [-1,1])for(let j=0;j<3;j++){if(side===1&&j===0)continue;const y=.28-j*.38,hip=v(side*.22,y,.64),knee=v(side*.67,y+(j===0?.38:j===2?-.3:.05),.35),foot=v(side*.95,knee.y+(j===0?.16:j===2?-.22:.06),.03);placeRod(rod(darkMat,.025),hip,knee);placeRod(rod(darkMat,.018),knee,foot);ellipsoid(knee,v(.035,.035,.035),darkMat);}\nconst plane=new THREE.Mesh(new THREE.PlaneGeometry(8,8),new THREE.MeshStandardMaterial({color:0xe9eef1,roughness:1}));plane.position.z=-.045;flyScene.add(plane);\nconst paperMesh=new THREE.Mesh(new THREE.PlaneGeometry(1.1,1.1),new THREE.MeshStandardMaterial({color:0xffffff,roughness:1}));paperMesh.position.set(1.03,.3,-.01);flyScene.add(paperMesh);\nconst co=rod(legMat,.032),femur=rod(legMat,.038),tibia=rod(legMat,.025),penRod=rod(darkMat,.018),knuckle=ellipsoid(v(0,0,0),v(.05,.05,.05),legMat),tipDot=ellipsoid(v(0,0,0),v(.025,.025,.025),darkMat);\nconst linePositions=new Float32Array((S-1)*6),inkGeometry=new THREE.BufferGeometry();inkGeometry.setAttribute(\'position\',new THREE.BufferAttribute(linePositions,3).setUsage(THREE.DynamicDrawUsage));inkGeometry.setDrawRange(0,0);flyScene.add(new THREE.LineSegments(inkGeometry,new THREE.LineBasicMaterial({color:0x233541})));\nconst contexts=Object.fromEntries([\'eye\',\'retina\',\'paper\',\'feedback\'].map(k=>[k,$(k).getContext(\'2d\')]));\nfor(const k of Object.keys(contexts)){ $(k).width=512;$(k).height=512; }\nconst native=document.createElement(\'canvas\');native.width=native.height=H;const nativeCtx=native.getContext(\'2d\'),raster=nativeCtx.createImageData(H,H);\nfunction paintImage(ctx,images,index,complete=true){const W=ctx.canvas.width;ctx.fillStyle=\'#08121e\';ctx.fillRect(0,0,W,W);if(index<0||!complete)return;const off=index*H*H;for(let i=0;i<H*H;i++){const b=images[off+i];raster.data[4*i]=raster.data[4*i+1]=raster.data[4*i+2]=b;raster.data[4*i+3]=255;}nativeCtx.putImageData(raster,0,0);ctx.imageSmoothingEnabled=false;ctx.drawImage(native,0,0,W,W);}\nfunction penXY(output,point){const i=(output*S+point)*3;return [a.traces[i],a.traces[i+1],a.traces[i+2]];}\nfunction paintTrace(ctx,output,point,head=true){const W=ctx.canvas.width;ctx.fillStyle=\'#fbfcfd\';ctx.fillRect(0,0,W,W);if(output<0)return;const px=x=>(x-.25)/1.1*W,py=y=>(.55-y)/1.1*W;ctx.strokeStyle=\'#1b3441\';ctx.lineWidth=Math.max(1.5,W/H);ctx.lineCap=\'round\';ctx.lineJoin=\'round\';for(let j=1;j<=point;j++){const p=penXY(output,j-1),q=penXY(output,j);if(q[2]<.5)continue;ctx.beginPath();ctx.moveTo(px(p[0]),py(p[1]));ctx.lineTo(px(q[0]),py(q[1]));ctx.stroke();}if(head&&point>=0){const p=penXY(output,point);ctx.fillStyle=point>0&&p[2]>=.5?\'#de8c33\':\'#178da2\';ctx.beginPath();ctx.arc(px(p[0]),py(p[1]),W/90,0,Math.PI*2);ctx.fill();}}\nconst gallery=[];for(let i=0;i<C;i++){const b=document.createElement(\'button\');b.className=\'thumb future\';b.setAttribute(\'aria-label\',\'Replay pen sample \'+(i+1));const c=document.createElement(\'canvas\');c.width=c.height=160;const label=document.createElement(\'span\');label.textContent=\'Sample \'+String(i+1).padStart(2,\'0\');b.append(c,label);$(\'gallery\').append(b);b.addEventListener(\'click\',()=>seekTo(data.outputs[i].start));gallery.push({button:b,ctx:c.getContext(\'2d\')});}\nfunction updateGallery(output,point){for(let i=0;i<C;i++){const g=gallery[i],before=i<output,now=i===output;g.button.classList.toggle(\'future\',i>output);g.button.classList.toggle(\'active\',now);const visible=before||now,p=before?S-1:point;if(g.point!==p||g.visible!==visible){paintTrace(g.ctx,visible?i:-1,p,false);g.point=p;g.visible=visible;}}}\nfunction paintRetina(frame){const ctx=contexts.retina,W=ctx.canvas.width;ctx.fillStyle=\'#08121e\';ctx.fillRect(0,0,W,W);for(let i=0;i<R;i++){const brightness=a.retina[frame*R+i]/65535;ctx.fillStyle=`rgba(91,230,242,${.06+.94*Math.sqrt(brightness)})`;ctx.beginPath();ctx.arc(a.uv[2*i]*W,a.uv[2*i+1]*W,Math.max(1.2,W/155),0,Math.PI*2);ctx.fill();}}\nfunction updateFly(output,point,kind){let q1=-.9,q2=1.8,contact=false;if(output>=0&&point>=0){const k=(output*S+point)*2;q1=a.joints[k];q2=a.joints[k+1];contact=kind===\'draw\'&&point>0&&penXY(output,point)[2]>=.5;}const z=contact?.20:.37,root=v(.23,.3,z),elbow=v(.23+data.links[0]*Math.cos(q1),.3+data.links[0]*Math.sin(q1),z),tip=v(elbow.x+data.links[1]*Math.cos(q1+q2),elbow.y+data.links[1]*Math.sin(q1+q2),z),end=v(tip.x,tip.y,z-.2);placeRod(co,v(.23,.3,.65),root);placeRod(femur,root,elbow);placeRod(tibia,elbow,tip);placeRod(penRod,tip,end);knuckle.position.copy(elbow);tipDot.position.copy(end);let vertices=0;for(let j=1;output>=0&&j<=point;j++){const p=penXY(output,j-1),q=penXY(output,j);if(q[2]<.5)continue;linePositions.set([.23+p[0],.3+p[1],.004,.23+q[0],.3+q[1],.004],vertices*3);vertices+=2;}inkGeometry.attributes.position.needsUpdate=true;inkGeometry.setDrawRange(0,vertices);inkGeometry.computeBoundingSphere();$(\'penState\').textContent=output<0?\'Observing · pen raised\':(contact?\'Pen down · drawing\':\'Pen up · moving\')+\' · sample \'+(output+1);}\nfunction eventAt(t){let lo=0,hi=events.length-1;while(lo<hi){const mid=Math.ceil((lo+hi)/2);if(events[mid].start<=t)lo=mid;else hi=mid-1;}return lo;}\nfunction stateAt(t){const ei=eventAt(t),e=events[ei],obs=data.observations[e.observation];let frame=obs.frames[0],point=-1,output=e.output;\n if(e.kind===\'look\'){point=output>=0?S-1:-1;const tick=Math.min(data.brain_steps,Math.floor((t-e.start)/e.duration*data.brain_steps));for(let j=0;j<obs.ticks.length;j++){if(obs.ticks[j]<=tick)frame=obs.frames[j];else break;}}else{frame=obs.frames[obs.frames.length-1];point=Math.min(S-1,Math.max(0,Math.floor((t-e.start)/e.duration*(S-1))));}\n return {event:ei,frame,observation:e.observation,output,point,kind:e.kind};}\nfunction drawNeurons(st){const mode=$(\'activityMode\').value,spikeOn=st.kind===\'look\';if(st.frame!==lastFrame||mode!==lastMode||st.kind!==current.kind){let active=0;const offset=st.frame*n;for(let i=0;i<n;i++){const sp=a.spikes[offset+i];const sg=a.signal[offset+i]/65535;const val=mode===\'signal\'?sg:mode===\'spikes\'?(spikeOn?Math.min(sp,1):0):a.spiking[i]?(spikeOn?Math.min(sp,1):0):sg;levels[i]=val;if(val>0)active++;}brainGeometry.attributes.level.needsUpdate=true;$(\'activityInfo\').textContent=st.kind===\'draw\'?\'Neural state held while the open-loop pen command plays\':active.toLocaleString()+\' active · recorded tick \'+data.frames[st.frame].global_tick;lastFrame=st.frame;lastMode=mode;}}\nfunction frameUpdate(){const st=stateAt(time),e=events[st.event],obs=data.observations[st.observation];drawNeurons(st);if(st.observation!==lastObservation){paintImage(contexts.eye,a.images,st.observation);$(\'eyeCaption\').textContent=(obs.source===\'prompt\'?\'Prompt character image\':\'Previous pen drawing\')+\' · \'+H+\' × \'+H+\' actual input pixels, enlarged\';lastObservation=st.observation;}if(st.frame!==lastRetinaFrame){paintRetina(st.frame);lastRetinaFrame=st.frame;}\n const drawKey=st.kind+\':\'+st.output+\':\'+st.point;if(drawKey!==lastDrawKey){paintTrace(contexts.paper,st.output,st.point);paintImage(contexts.feedback,a.feedback,st.output,st.output>=0&&(st.kind===\'look\'||st.point===S-1));updateFly(st.output,st.point,st.kind);updateGallery(st.output,st.point);lastDrawKey=drawKey;}\n $(\'exposureLabel\').textContent=\'Exposure \'+(st.observation+1)+\' / \'+O+\' · \'+(obs.source===\'prompt\'?\'prompt\':\'pen feedback\');$(\'phase\').textContent=st.kind===\'look\'?\'Seeing · neural dynamics\':\'Drawing · pen trajectory\';$(\'frameInfo\').textContent=\'Recorded neural frame \'+(st.frame+1)+\' / \'+F+\' · \'+(st.kind===\'look\'?\'spikes are events in the preceding recorded tick interval\':\'no neural ticks elapse during this pen replay\')+\' · replay seconds are not biological seconds\';$(\'time\').textContent=clock(time)+\' / \'+clock(total);$(\'seek\').value=time;current=st;}\nfunction clock(t){return Math.floor(t/60)+\':\'+String(Math.floor(t%60)).padStart(2,\'0\');}\nfunction resize(){for(const [r,host] of [[brainRenderer,$(\'brainStage\')],[flyRenderer,$(\'flyStage\')]]){const w=host.clientWidth,h=host.clientHeight;r.setPixelRatio(Math.min(Number($(\'quality\').value),4096/Math.max(w,h)));r.setSize(w,h,false);}flyCamera.aspect=$(\'flyStage\').clientWidth/$(\'flyStage\').clientHeight;flyCamera.updateProjectionMatrix();dirty=true;}\nfunction renderBrain(){const w=brainRenderer.domElement.clientWidth,h=brainRenderer.domElement.clientHeight;brainMaterial.uniforms.pointScale.value=brainRenderer.getPixelRatio();brainMaterial.uniforms.gain.value=Number($(\'gain\').value);brainRenderer.setScissorTest(false);brainRenderer.setViewport(0,0,w,h);brainRenderer.clear();if($(\'brainView\').value===\'dual\'){brainRenderer.setScissorTest(true);for(const [cam,x] of [[cameraFront,0],[cameraSide,w/2]]){const aspect=(w/2)/h;const halfHeight=1.1*Math.max(brainExtent[cam===cameraFront?1:2],brainExtent[0]/aspect,.01);cam.left=-halfHeight*aspect;cam.right=halfHeight*aspect;cam.top=halfHeight;cam.bottom=-halfHeight;cam.updateProjectionMatrix();brainRenderer.setViewport(x,0,w/2,h);brainRenderer.setScissor(x,0,w/2,h);brainRenderer.render(brainScene,cam);}brainRenderer.setScissorTest(false);}else{cameraOrbit.aspect=w/h;cameraOrbit.updateProjectionMatrix();brainRenderer.render(brainScene,cameraOrbit);}}\nfunction render(){frameUpdate();renderBrain();flyRenderer.render(flyScene,flyCamera);dirty=false;}\nfunction seekTo(t){time=Math.max(0,Math.min(total,t));dirty=true;render();}\n$(\'play\').onclick=()=>{if(time>=total)time=0;playing=!playing;$(\'play\').textContent=playing?\'Pause\':\'Play\';dirty=true;};$(\'seek\').addEventListener(\'input\',e=>seekTo(Number(e.target.value)));$(\'speed\').addEventListener(\'input\',e=>{speed=Number(e.target.value);$(\'speedValue\').textContent=speed.toFixed(1)+\'×\';});\n$(\'previous\').onclick=()=>{const earlier=data.observations.filter(o=>o.start<time-.02);seekTo(earlier.length?earlier[earlier.length-1].start:0);};$(\'next\').onclick=()=>{const next=data.observations.find(o=>o.start>time+.02);seekTo(next?next.start:total);};\nfor(const id of [\'activityMode\',\'gain\',\'brainView\'])$(id).addEventListener(\'input\',()=>{lastFrame=-1;dirty=true;const dual=$(\'brainView\').value===\'dual\';$(\'brainLabel\').textContent=dual?\'X–Z\':\'3D · drag to orbit, scroll to zoom\';$(\'brainLabelRight\').style.display=$(\'brainDivider\').style.display=dual?\'\':\'none\';});$(\'quality\').addEventListener(\'change\',resize);$(\'resetFly\').onclick=resetFly;\n$(\'fullBrain\').onclick=async()=>{try{if(document.fullscreenElement)await document.exitFullscreen();else await $(\'brainCard\').requestFullscreen();}catch(e){$(\'notes\').textContent=\'Fullscreen is unavailable in this notebook frame; open the saved HTML.\';}};\n$(\'saveBrain\').onclick=()=>{try{const w=$(\'brainStage\').clientWidth,h=$(\'brainStage\').clientHeight,ratio=brainRenderer.getPixelRatio();brainRenderer.setPixelRatio(3840/w);renderBrain();brainRenderer.domElement.toBlob(blob=>{if(!blob)return;const url=URL.createObjectURL(blob),link=document.createElement(\'a\');link.href=url;link.download=\'flylm3-neurons-4k.png\';link.click();setTimeout(()=>URL.revokeObjectURL(url),1000);},\'image/png\');brainRenderer.setPixelRatio(ratio);brainRenderer.setSize(w,h,false);dirty=true;}catch(e){fail(e);}};\ndocument.addEventListener(\'visibilitychange\',()=>{lastRAF=0;if(document.hidden){playing=false;$(\'play\').textContent=\'Play\';}});new ResizeObserver(resize).observe($(\'brainStage\'));new ResizeObserver(resize).observe($(\'flyStage\'));\n$(\'seek\').max=total;$(\'summary\').textContent=n.toLocaleString()+\' neurons · \'+R.toLocaleString()+\' mapped receptors · \'+data.motor_count+\' foreleg motors\';$(\'notes\').textContent=\'16-bit synaptic brightness and recorded spike counts. Graded cells do not emit spike events in this model. The fly body and pen lift are illustrative; the two planar joint angles and contact sequence are recorded model outputs. All viewer assets are embedded; no network connection is needed.\';\n$(\'load\').style.display=\'none\';$(\'play\').disabled=false;resize();render();\nfunction animate(now){if(lastRAF&&playing){time+=Math.min((now-lastRAF)/1000,.1)*speed;if(time>=total){if($(\'loop\').checked)time%=total;else{time=total;playing=false;$(\'play\').textContent=\'Play\';}}dirty=true;}lastRAF=now;if(dirty)render();requestAnimationFrame(animate);}requestAnimationFrame(animate);\nwindow.addEventListener(\'beforeunload\',()=>{brainRenderer.dispose();flyRenderer.dispose();});\ndocument.body.dataset.ready=\'true\';\n}catch(e){fail(e);}\n})();\n</script></body></html>\n'

def packed_array(array,dtype,tag):
    a=np.ascontiguousarray(array,dtype=dtype)
    return dict(type=tag,shape=list(a.shape),data=base64.b64encode(gzip.compress(a.tobytes(),compresslevel=6,mtime=0)).decode('ascii'))

def save_replay(recording,layout,path):
    arrays={}
    for key in ['signal','spikes','retina']: arrays[key]=packed_array(recording[key],'<u2','u16')
    for key in ['images','feedback']: arrays[key]=packed_array(recording[key],np.uint8,'u8')
    for key in ['traces','joints']: arrays[key]=packed_array(recording[key],'<f4','f32')
    arrays['positions']=packed_array(layout['positions'],'<f4','f32')
    arrays['roles']=packed_array(layout['roles'],np.uint8,'u8')
    arrays['spiking']=packed_array(model.spiking_mask.cpu().numpy(),np.uint8,'u8')
    arrays['uv']=packed_array(graph['uv'],'<f4','f32')
    payload={key:recording[key] for key in ['frames','observations','outputs','events','duration','brain_steps']}
    payload.update(neurons=model.n,receptors=len(model.inputs),motor_count=len(model.outputs),
        image_size=CFG['image_size'],pen_steps=CFG['pen_steps'],links=CFG['link_lengths'],arrays=arrays)
    # JSON is data, including any user-supplied strings; never interpolate it as HTML.
    serialized=json.dumps(payload,separators=(',',':')).replace('<','\u003c').replace('>','\u003e').replace('&','\u0026')
    viewer=PLAYER_HTML.replace('__THREE_JS__',THREE_JS.replace('</script','<'+chr(92)+'/script')).replace('__PAYLOAD__',serialized)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(viewer,encoding='utf-8')
    return path

def show_replay(path,height=1540):
    # srcdoc keeps this self-contained in notebook outputs; escaping prevents attribute injection.
    markup=Path(path).read_text(encoding='utf-8')
    display(HTML('<iframe title="FlyLM-3 ABC copy replay" sandbox="allow-scripts allow-downloads allow-same-origin" '
        'allow="fullscreen" allowfullscreen style="width:100%;height:'+str(int(height))+'px;border:0" '
        'srcdoc="'+html.escape(markup,quote=True)+'"></iframe>'))
    display(FileLink(str(path),result_html_prefix='Offline replay: '))

## 7. Choose letters and run the replay

Start with `LETTERS='ABC'` and `COPIES_PER_LETTER=1`. Each letter is copied independently.
Set copies to 4 to test repeated self-feedback. Repeated copies may drift because training
used only canonical glyphs; the replay displays the actual drift rather than substituting
recognized letters. `MAX_NEURAL_FRAMES` bounds recording memory.

In [ ]:
LETTERS = 'ABC'
COPIES_PER_LETTER = 1
MAX_NEURAL_FRAMES = 180
LOOK_REPLAY_SECONDS = 0.6
DRAW_REPLAY_SECONDS = 2.0

recording = record_abc_copy(model, LETTERS, COPIES_PER_LETTER, MAX_NEURAL_FRAMES,
    look_seconds=LOOK_REPLAY_SECONDS, draw_seconds=DRAW_REPLAY_SECONDS)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
REPLAY_PATH = save_replay(recording, layout, OUTPUT_ROOT/f'flylm3_abc_copy_{stamp}.html')
print(f"Recorded {recording['neural_ticks']} actual ticks and {len(recording['outputs'])} pen drawings.")
print('Each exposure resets state. Checkpoint phase:', model.checkpoint_phase)
show_replay(REPLAY_PATH)

## 7. Save the numerical recording (optional)

The HTML is enough for playback. This separate export preserves the 16-bit activity, spike
counts, retinal responses, input pixels, pen positions, joint angles, neuron IDs and coordinate
provenance for later inspection. It contains no recognized/generated text.

In [ ]:
SAVE_NUMERICAL_RECORDING = False
if SAVE_NUMERICAL_RECORDING:
    arrays={k:recording[k] for k in ['signal','spikes','retina','images','feedback','traces','joints']}
    np.savez_compressed(REPLAY_PATH.with_suffix('.npz'),**arrays,
        body_ids=layout['body_ids'],xyz=layout['xyz'],coordinate_source=layout['source'])
    meta={k:v for k,v in recording.items() if not isinstance(v,np.ndarray)}
    meta.update(signal_encoding='uint16 / 65535; synaptic signal clipped to [0,1]',
        spike_encoding='uint16 count of model spike events since previous sample within the exposure',
        retinal_encoding='uint16 / 65535; filtered receptor luminance',
        time='dimensionless neural ticks; separate artificial replay seconds')
    REPLAY_PATH.with_suffix('.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Saved numerical recording beside the HTML.')

## Interpretation and validation

Correct A/B/C copies plus failed blank/motor-clamp controls show that this checkpoint's
motor-only head uses the visual input on these three images. A frozen-circuit checkpoint
can pass that test without learning new synaptic strengths. Inspect the training report
for whether end-to-end optimization helped. Self-feedback and new visual styles are
outside the canonical-image overfit criterion.

This notebook uses the existing FlyLM-3 anatomical coordinate archive and offline viewer
with the same provenance and visualization limits. It models no physical leg forces,
muscles or real fly behavior. Numerical and replay validation are recorded below after testing.

## Model and visualization limits

- Spike events are from the saved approximate model; graded cells retain graded dynamics.
  “Spikes + graded signal” shows events for spiking cells and continuous signal for graded cells.
  During a drawing phase there are no new neural ticks; the viewer suppresses spike flashes.
- Neuron coordinates are real MaleCNS soma/skeleton representative positions. They are not
  an entire neuron mesh or a reconstructed fly body. The procedural body gives context for
  the leg movement and is explicitly illustrative. The pen's planar link lengths and joint
  angles come from the checkpoint; its independent lift is shown as a vertical offset.
- Each output is one open-loop trajectory. Page placement is a gallery of samples; there is
  no inferred word spacing, automatic newline layout, or recognized text substituted for ink.
- Low-quality checkpoints may draw unreadable marks or blank images. The viewer shows those
  outputs faithfully. Neither high visual resolution nor replay speed changes model quality.

References: [MaleCNS](https://male-cns.janelia.org/download/),
[doomfly visual interface](https://github.com/nftechie/doomfly/blob/main/doom/README.md),
[female VNC reconstruction](https://pmc.ncbi.nlm.nih.gov/articles/PMC11348827/),
[leg/wing premotor organization](https://www.nature.com/articles/s41586-024-07600-z),
[Three.js documentation](https://threejs.org/docs/).